# 🎵 DC v2.0 — Build Android APK

This notebook builds the DC YouTube MP3 Downloader APK.

**Run each cell in order.** Total build time: ~15 minutes.

## Step 1: Install System Dependencies (~1 min)

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq \
    build-essential git ffmpeg libffi-dev libssl-dev \
    python3-venv zip unzip autoconf automake libtool \
    pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev \
    cmake openjdk-17-jdk lld
print('\n✅ System dependencies installed')

## Step 2: Install Buildozer & Cython (~30 sec)

In [ ]:
!pip install -q buildozer==1.5.0 cython==0.29.33
!buildozer --version
!cython --version
print('\n✅ Buildozer & Cython installed')

## Step 3: Clone Repository (~10 sec)

In [ ]:
%cd /content
!rm -rf DC
!git clone https://github.com/SONUVERMA11/DC.git
%cd /content/DC/android_app
!ls -la
print('\n✅ Repository cloned')

## Step 4: Build APK ⏳ (~15 min)

**You will see live build output scrolling.** This is normal — wait for it to finish.

The build downloads SDKs, NDK, compiles Python, and packages everything.

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# Verify Java
!java -version
print('\n🔨 Starting build... (this takes 10-20 minutes, output will scroll below)\n')
print('=' * 60)

# Build with live output — NO pipe, so you see progress in real-time
# 'yes' auto-accepts any SDK license prompts
!cd /content/DC/android_app && yes | buildozer -v android debug

print('\n' + '=' * 60)
print('🏁 Build process completed!')

## Step 5: Find & Download APK

In [ ]:
import glob, os
from google.colab import files

# Search for APK in all possible locations
apk_files = glob.glob('/content/DC/android_app/bin/*.apk')
if not apk_files:
    apk_files = glob.glob('/content/DC/android_app/.buildozer/**/bin/*.apk', recursive=True)
if not apk_files:
    import subprocess
    result = subprocess.run(['find', '/content/DC', '-name', '*.apk', '-type', 'f'], capture_output=True, text=True)
    apk_files = [f.strip() for f in result.stdout.strip().split('\n') if f.strip()]

if apk_files:
    for apk in apk_files:
        size_mb = os.path.getsize(apk) / (1024 * 1024)
        print(f'✅ Found APK: {os.path.basename(apk)} ({size_mb:.1f} MB)')
        files.download(apk)
else:
    print('❌ No APK found. Run the debug cell below.')

## 🔍 Debug (only if build failed)

In [ ]:
import os
log_paths = [
    '/content/DC/android_app/.buildozer/logs/buildozer.log',
]
for lp in log_paths:
    if os.path.exists(lp):
        print(f'=== LAST 100 LINES OF {lp} ===')
        !tail -100 {lp}
        print(f'\n=== ERRORS ===')
        !grep -i -E '(error|exception|failed|traceback)' {lp} | tail -20
        break
else:
    print('No buildozer log found')
    !find /content/DC -name '*.log' -type f